# Pump RUL Regressor Training

This notebook trains 5 separate XGBoost regressors to predict Remaining Useful Life (RUL) for each failure mode:
- PUMP_RUL_BEARING_WEAR
- PUMP_RUL_CAVITATION
- PUMP_RUL_VALVE_FAILURE
- PUMP_RUL_OVERHEATING
- PUMP_RUL_SEAL_LEAK

Each model is trained ONLY on data from its specific failure mode's degradation window.

## Required Packages

Add these packages via the **Packages** dropdown in Snowsight:
- `snowflake-ml-python`
- `xgboost`
- `scikit-learn`
- `pandas`
- `numpy`

In [ ]:
# Import packages
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col
from snowflake.ml.registry import Registry
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

session = get_active_session()
session.use_database("PDM_DEMO")
session.use_schema("ML")

reg = Registry(session=session, database_name="PDM_DEMO", schema_name="ML")
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

In [ ]:
# Define feature columns and failure modes
FEATURE_COLS = [
    'SUCTION_PRESSURE', 'DISCHARGE_PRESSURE', 'FLOW_RATE', 'MOTOR_CURRENT',
    'PUMP_SPEED', 'BEARING_TEMP', 'CASING_TEMP', 'VIBRATION_RMS', 
    'VALVE_POSITION', 'LEAK_RATE', 'PRESSURE_DIFF', 'POWER_PROXY',
    'VIBRATION_1H_MEAN', 'VIBRATION_1H_STD', 'BEARING_TEMP_1H_MEAN', 
    'BEARING_TEMP_1H_STD', 'LEAK_RATE_1H_MEAN', 'LEAK_RATE_1H_STD',
    'SUCTION_1H_MEAN', 'SUCTION_1H_STD', 'FLOW_1H_MEAN', 'FLOW_1H_STD',
    'VIBRATION_24H_MEAN', 'VIBRATION_24H_STD', 'BEARING_TEMP_24H_MEAN',
    'BEARING_TEMP_24H_STD', 'LEAK_RATE_24H_MEAN', 'LEAK_RATE_24H_STD',
    'SUCTION_24H_MEAN', 'SUCTION_24H_STD', 'FLOW_24H_MEAN', 'FLOW_24H_STD',
    'VALVE_24H_MEAN', 'VALVE_24H_STD', 'CASING_TEMP_24H_MEAN',
    'MOTOR_CURRENT_24H_MEAN', 'VIBRATION_TREND_24H', 'TEMP_TREND_24H',
    'LEAK_TREND_24H', 'SUCTION_TREND_24H', 'FLOW_TREND_24H'
]

FAILURE_MODES = ['BEARING_WEAR', 'CAVITATION', 'VALVE_FAILURE', 'OVERHEATING', 'SEAL_LEAK']
print(f"Training {len(FAILURE_MODES)} RUL models")

In [ ]:
# Load all training data with RUL labels
print("Loading training data...")
df = session.table("PUMP_TRAINING_DATA").filter(
    (col("VIBRATION_TREND_24H").is_not_null()) & 
    (col("RUL_DAYS").is_not_null()) &
    (col("RUL_DAYS") > 0)
).select(FEATURE_COLS + ['STATE', 'RUL_DAYS', 'IS_TRAIN']).to_pandas()

print(f"Total rows with RUL: {len(df):,}")
print(f"\nRows per failure mode:")
print(df['STATE'].value_counts())

In [ ]:
# Train RUL model for each failure mode
results = []

for mode in FAILURE_MODES:
    print(f"\n{'='*50}")
    print(f"Training RUL model for: {mode}")
    print(f"{'='*50}")
    
    mode_df = df[df['STATE'] == mode].copy()
    train_df = mode_df[mode_df['IS_TRAIN'] == True]
    test_df = mode_df[mode_df['IS_TRAIN'] == False]
    
    print(f"Train: {len(train_df):,} rows, Test: {len(test_df):,} rows")
    
    if len(train_df) < 100:
        print(f"Skipping {mode} - insufficient training data")
        continue
    
    X_train = train_df[FEATURE_COLS].fillna(0).values
    y_train = train_df['RUL_DAYS'].values
    X_test = test_df[FEATURE_COLS].fillna(0).values
    y_test = test_df['RUL_DAYS'].values
    
    regressor = xgb.XGBRegressor(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    )
    
    regressor.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    y_pred = regressor.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"MAE: {mae:.2f} days, R2: {r2:.4f}")
    
    model_name = f"PUMP_RUL_{mode}"
    sample_input = pd.DataFrame([X_train[0]], columns=FEATURE_COLS)
    
    mv = reg.log_model(
        regressor,
        model_name=model_name,
        version_name="V1",
        sample_input_data=sample_input,
        conda_dependencies=["xgboost"],
        target_platforms=["WAREHOUSE"],
        comment=f"RUL regressor for {mode} failures"
    )
    
    print(f"Registered: {mv.model_name} version {mv.version_name}")
    
    results.append({
        'model': model_name,
        'train_rows': len(train_df),
        'test_rows': len(test_df),
        'mae': mae,
        'r2': r2
    })

In [ ]:
# Summary of all RUL models
print("\n" + "="*60)
print("RUL MODEL TRAINING SUMMARY")
print("="*60)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Verify all models registered
print("\nModels in registry:")
session.sql("SHOW MODELS LIKE 'PUMP%' IN SCHEMA PDM_DEMO.ML").show()